# NEO — Exploratory Notebook

**ArXivist-generated.** Paper: [arXiv:2607.24538](https://arxiv.org/abs/2607.24538)

Visualizes three things worth looking at directly:
1. The synthetic stand-in scene rendered from a training viewpoint
2. What the two-stage neural-field resampling (Sec. II-A) actually does to ray samples
3. Table III ablation results (if you've run `run_ablation.py`)

Requires no downloads. Uses the same synthetic scene as the primary notebook.

In [ ]:
import sys, os
repo_root = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(repo_root, "src"))
import numpy as np
import torch
import matplotlib.pyplot as plt

from neo_nerf_editing.data.synthetic_scene import ToySceneGenerator
from neo_nerf_editing.data.rays import fixed_scan_trajectory
from neo_nerf_editing.models.object_removal import RayBoxExcluder, TwoStageResampler
from neo_nerf_editing.models.language_field import OrientedBox

print("Imports OK.")

## 1. The synthetic stand-in scene

Ground-truth analytic rendering (not a learned NeRF) of one training view.

In [ ]:
scene_gen = ToySceneGenerator()
scene = scene_gen.generate_scene(seed=0, objects=["soup_can", "lego_brick"])
traj = fixed_scan_trajectory(n_views=1, radius=4.0, height=2.0)
render = scene_gen.render_views(scene, traj, H=96, W=96, near=2.0, far=8.0)

plt.figure(figsize=(4, 4))
plt.imshow(render["images"][0].numpy())
plt.title("Synthetic scene, ground-truth render")
plt.axis("off")
plt.show()

for o in scene.objects:
    print(f"  object={o.name:12s} center={o.center.round(2)} half_extents={o.half_extents.round(2)}")

## 2. Neural field resampling (Sec. II-A, Fig. 4)

Compares standard uniform ray sampling against the paper's box-exclusion resampling: samples
that would have fallen inside the removed-object's bounding box are reallocated to the rest of
the ray instead.

In [ ]:
box = OrientedBox(center=np.array([0.0, 0.0, 4.0], dtype=np.float32), yaw=0.0,
                   extents=np.array([0.5, 0.5, 0.5], dtype=np.float32))
rays_o = torch.zeros(1, 3)
rays_d = torch.tensor([[0.0, 0.0, 1.0]])

excluder, resampler = RayBoxExcluder(), TwoStageResampler()
hits = excluder.intersect(rays_o, rays_d, box, near=1.0, far=8.0)
seg_near, seg_far = excluder.build_exclusion_intervals(1.0, 8.0, hits)
t_standard = torch.linspace(1.0, 8.0, 40)
t_resampled = resampler.uniform_resample(seg_near, seg_far, n_uniform=40)[0]

fig, axes = plt.subplots(2, 1, figsize=(8, 3), sharex=True)
axes[0].scatter(t_standard.numpy(), np.zeros_like(t_standard), s=10)
axes[0].axvspan(hits[0, 0].item(), hits[0, 1].item(), color="orange", alpha=0.3, label="excluded box region")
axes[0].set_title("Standard uniform sampling (paper's Fig. 4a / DFF-style Fig. 4b)")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_yticks([])

axes[1].scatter(t_resampled.numpy(), np.zeros_like(t_resampled.numpy()), s=10, color="green")
axes[1].axvspan(hits[0, 0].item(), hits[0, 1].item(), color="orange", alpha=0.3)
axes[1].set_title("NEO's resampling (Fig. 4c): samples reallocated away from the box")
axes[1].set_yticks([])
axes[1].set_xlabel("t (distance along ray)")
plt.tight_layout()
plt.show()

n_inside = box.contains_torch(rays_o + t_resampled[:, None] * rays_d).sum().item()
print(f"Resampled points landing inside the excluded box: {n_inside} / {len(t_resampled)} (expect 0)")

## 3. Ablation results (Table III variants)

Run `python run_ablation.py --config configs/config.yaml --out-dir runs/ablation --debug` first
to populate `runs/ablation/ablation_results.json` (or point `RESULTS_PATH` below at your own run).

In [ ]:
import json

RESULTS_PATH = os.path.join(repo_root, "runs", "ablation", "ablation_results.json")

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        ablation_results = json.load(f)
    variants = list(ablation_results.keys())
    psnr_out = [ablation_results[v]["psnr_out"] for v in variants]

    plt.figure(figsize=(7, 3.5))
    plt.bar(variants, psnr_out, color="steelblue")
    plt.ylabel("PSNR (Out region, dB)")
    plt.title("Table III ablation: PSNR (Out) by variant")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print(f"No ablation results found at {RESULTS_PATH}.")
    print("Run: python run_ablation.py --config configs/config.yaml --out-dir runs/ablation --debug")
    print("...then re-run this cell.")